# PEFT full run


| Phase | Runtime | What runs | Depends on |
|------|---------|-----------|------------|
| 1 | **generation** (`requirements.txt`) | `peft_sweep`: train 4 cells x 3 epochs, eval_loss pre-filter, generate + score the surviving `(r, lr, epoch)` candidates | -- |
| 2 | **COMET** (`requirements-comet.txt`) | `peft_verify` (COMET + judge Phi), **freeze** the adapter into `configs/peft_qwen.yaml` | Phase 1 sweep result |
| 3 | **generation** (`requirements.txt`) | the `peft` rung on full val + chrF/BLEU + stylometrics + judge Phi over the six-condition ladder | frozen adapter |
| 4 | **COMET** (`requirements-comet.txt`) | ladder COMET + paired bootstrap | Phase 3 outputs |


---
## Phase 1 -- generation runtime - train the grid, score the candidates

In [1]:
# Confirm the GPU: training is 7B bf16 + LoRA on all-linear, generation reloads the
# same base per candidate (~15 GB weights). Anything under ~40 GB will thrash or OOM.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4090, 24564 MiB


In [8]:
import os
if not os.path.exists('manage.py') and not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/peft-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

[Errno 2] No such file or directory: 'Style-Aware-MT'
/home/prnamhr/projects/Style-Aware-MT/notebooks/Style-Aware-MT
a9eb0de


In [3]:
# Generation stack.
!pip install -r requirements.txt

# torchvision/torchaudio ship a pinned-torch ABI that the requirements torch breaks;
# this pipeline is text-only, so drop them rather than resolve them.
!pip uninstall -y torchvision torchaudio

In [4]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)

2.12.0+cu130 True 13.0


In [9]:
# Qwen2.5-7B-Instruct is not gated, but a token avoids anonymous-download throttling.
import getpass, os
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN (blank to skip): ')
print('HF_TOKEN set for this kernel:', 'yes' if os.environ.get('HF_TOKEN') else 'no')

HF_TOKEN set for this kernel: yes


In [ ]:
import yaml
from pathlib import Path

CONFIG = 'configs/peft_sweep.yaml'
cfg = yaml.safe_load(Path(CONFIG).read_text())
t = cfg['peft']['train']

n_val   = sum(1 for line in open(cfg['data']['eval_file'])  if line.strip())
n_train = sum(1 for line in open(cfg['data']['train_file']) if line.strip())

print(f"base            {cfg['generator']['model']}")
print(f"train / eval    {n_train} / {n_val}  ({cfg['data']['eval_file']})")
print(f"grid            {len(cfg['sweep']['grid'])} cells x {t['num_train_epochs']} epochs")
print(f"load_best       {t.get('load_best_model_at_end')}   save_total_limit {t.get('save_total_limit')}")
print(f"target_modules  {cfg['peft']['lora']['target_modules']}   4bit {cfg['peft']['load_in_4bit']}")

assert Path(cfg['data']['eval_file']).stem == 'val',  'sweep must run on val, never test'
assert cfg['data']['limit'] is None,                  'smoke limit leaked into the full run'
assert t['num_train_epochs'] == 3,                    'epoch is a tuned axis; expected 3'
assert t.get('load_best_model_at_end') is False,      'load_best collapses the epoch axis'
assert t.get('save_total_limit') is None,             'must keep every epoch checkpoint'
assert len(cfg['sweep']['grid']) == 4,                'expected the 4-cell Phase 2 grid'
assert Path(cfg['register']['centroid_file']).exists(), 'register centroid missing'
print('\npre-flight OK')

/workspace/Style-Aware-MT/notebooks/Style-Aware-MT
base            Qwen/Qwen2.5-7B-Instruct
train / eval    10860 / 1323  (data/splits/val.jsonl)
grid            4 cells x 3 epochs
load_best       False   save_total_limit None
target_modules  all-linear   4bit False

pre-flight OK


In [4]:
# Quarantine any peft_sweep generation that is not a full-val file (see the note above).
import json, shutil
from pathlib import Path

quarantine = Path('archive/peft-smoke')
stale = []
for p in sorted(Path('outputs/peft_sweep').glob('*_val.jsonl')):
    n = sum(1 for line in p.open() if line.strip())
    if n != n_val:
        stale.append((p, n))

# results/peft_sweep_val.json is rewritten by the sweep, but a crash mid-run would leave
# the smoke ranking for peft_verify to consume. Quarantine it with the rest.
smoke_result = Path('results/peft_sweep_val.json')
if smoke_result.exists():
    cells = json.loads(smoke_result.read_text()).get('cells', [])
    if any(c.get('n') != n_val for c in cells):
        stale.append((smoke_result, cells[0].get('n') if cells else None))

if stale:
    quarantine.mkdir(parents=True, exist_ok=True)
    for p, n in stale:
        shutil.move(str(p), quarantine / p.name)
        print(f'quarantined {p}  (n={n}, expected {n_val}) -> {quarantine / p.name}')
else:
    print(f'no stale artifacts: every peft_sweep file present is full-val (n={n_val})')

no stale artifacts: every peft_sweep file present is full-val (n=1323)


In [5]:
# The grid plan, trained nothing. Read the output before committing the GPU-hours.
!python3 manage.py peft_sweep --config configs/peft_sweep.yaml --dry-run


PEFT sweep: 4 cell(s) x 3 epoch(s) kept = candidate grid

cell     r  alpha        lr   output_dir (checkpoints: epoch 1..3)
------------------------------------------------------------------------
1       16     32      2e-4   models/peft_lora_r16_lr2e-4 (anchor)
2        8     16      2e-4   models/peft_lora_r8_lr2e-4
3       32     64      2e-4   models/peft_lora_r32_lr2e-4
4       16     32      1e-4   models/peft_lora_r16_lr1e-4

--dry-run: no training performed.


### The sweep

In [8]:
!python3 manage.py peft_sweep --config configs/peft_sweep.yaml --epochs-keep 2 --adequacy-margin 1.0


PEFT sweep: 4 cell(s) x 3 epoch(s) kept = candidate grid

cell     r  alpha        lr   output_dir (checkpoints: epoch 1..3)
------------------------------------------------------------------------
1       16     32      2e-4   models/peft_lora_r16_lr2e-4 (anchor)
2        8     16      2e-4   models/peft_lora_r8_lr2e-4
3       32     64      2e-4   models/peft_lora_r32_lr2e-4
4       16     32      1e-4   models/peft_lora_r16_lr1e-4


[cell 1/4] r=16 lr=2e-4 -> models/peft_lora_r16_lr2e-4
skip training models/peft_lora_r16_lr2e-4: epoch manifest present (--overwrite to retrain)

[cell 2/4] r=8 lr=2e-4 -> models/peft_lora_r8_lr2e-4
skip training models/peft_lora_r8_lr2e-4: epoch manifest present (--overwrite to retrain)

[cell 3/4] r=32 lr=2e-4 -> models/peft_lora_r32_lr2e-4
skip training models/peft_lora_r32_lr2e-4: epoch manifest present (--overwrite to retrain)

[cell 4/4] r=16 lr=1e-4 -> models/peft_lora_r16_lr1e-4
skip training models/peft_lora_r16_lr1e-4: epoch manifest present 

### Did the grid actually train?

Per cell: three distinct epoch checkpoints, no duplicate step (the e3-duplicate bug),
and an `eval_loss` that moves. A flat trajectory means no parameter updates reached
the model -- stop and fix that before spending Phase 2's judge budget.

In [10]:
import json
from pathlib import Path
from src.peft.sweep import _cell_dir

for cell in cfg['sweep']['grid']:
    d = _cell_dir(cfg['sweep']['output_base'], int(cell['r']), float(cell['lr']))
    man_path = d / 'epoch_checkpoints.json'
    if not man_path.exists():
        print(f'{d.name}: NO MANIFEST -- cell did not finish training')
        continue
    man = json.loads(man_path.read_text())
    epochs = [m['epoch'] for m in man]
    steps  = [m['step']  for m in man]
    losses = [m['eval_loss'] for m in man]
    ok = (
        len(man) == 3
        and epochs == [1, 2, 3]
        and len(set(steps)) == 3
        and all(m['checkpoint'] and Path(m['checkpoint']).exists() for m in man)
        and len({round(x, 6) for x in losses}) > 1
    )
    traj = '  '.join(f'e{e} {x:.4f}' for e, x in zip(epochs, losses))
    print(f"{'OK  ' if ok else 'FAIL'} {d.name:<26} steps {steps}  {traj}")
    if not ok:
        print(f'       epochs={epochs} distinct_steps={len(set(steps))} '
              f'distinct_losses={len({round(x, 6) for x in losses})}')

OK   peft_lora_r16_lr2e-4       steps [679, 1358, 2037]  e1 1.5312  e2 1.5601  e3 1.7814
OK   peft_lora_r8_lr2e-4        steps [679, 1358, 2037]  e1 1.5336  e2 1.5583  e3 1.7119
OK   peft_lora_r32_lr2e-4       steps [679, 1358, 2037]  e1 1.5398  e2 1.5733  e3 1.8708
OK   peft_lora_r16_lr1e-4       steps [679, 1358, 2037]  e1 1.5379  e2 1.5553  e3 1.6761


In [13]:
# The proxy ranking, and the pick the sweep recommends (free/local proxies only).
import json
sweep = json.load(open('results/peft_sweep_val.json'))
rows  = sweep['cells']

print(f"{'tag':<24} {'r':>3} {'lr':>7} {'ep':>3} {'n':>5} {'chrF':>6} {'reg_fit':>8} {'eval_loss':>10}")
print('-' * 72)
for r in sorted(rows, key=lambda r: r['register_fit']):
    mark = '  <== proxy pick' if r['tag'] == sweep['recommended']['tag'] else ''
    print(f"{r['tag']:<24} {r['r']:>3} {r['lr']:>7g} {r['epoch']:>3} {r['n']:>5} "
          f"{r['chrF']:>6} {r['register_fit']:>8} {r['eval_loss']:>10.4f}{mark}")

# Every candidate must be scored on the same full-val set, or the ranking is not a ranking.
assert {r['n'] for r in rows} == {n_val}, f"candidates scored on mixed sizes: {sorted({r['n'] for r in rows})}"
print(f'\nall {len(rows)} candidates scored on n={n_val}')

tag                        r      lr  ep     n   chrF  reg_fit  eval_loss
------------------------------------------------------------------------
peft_r32_lr2e-4_e2        32  0.0002   2  1323  41.58   1.0265     1.5733  <== proxy pick
peft_r16_lr2e-4_e2        16  0.0002   2  1323  41.73   1.0386     1.5601
peft_r16_lr1e-4_e2        16  0.0001   2  1323  41.66   1.0449     1.5553
peft_r8_lr2e-4_e2          8  0.0002   2  1323  41.57   1.0616     1.5583
peft_r32_lr2e-4_e1        32  0.0002   1  1323  42.18   1.0736     1.5398
peft_r8_lr2e-4_e1          8  0.0002   1  1323   41.4   1.0776     1.5336
peft_r16_lr2e-4_e1        16  0.0002   1  1323  41.96   1.0829     1.5312
peft_r16_lr1e-4_e1        16  0.0001   1  1323   41.1   1.1015     1.5379

all 8 candidates scored on n=1323


---
## Phase 2 -- COMET runtime - verify + freeze

In [14]:
# Phase 2 context (re-derived from disk; safe after a runtime restart).
import json, yaml
from pathlib import Path
from src.peft.sweep import ranked_cells

CONFIG    = 'configs/peft_sweep.yaml'
cfg       = yaml.safe_load(Path(CONFIG).read_text())
n_val     = sum(1 for line in open(cfg['data']['eval_file']) if line.strip())
sweep_dir = Path(cfg['output']['dir']) / 'peft_sweep'
sweep     = json.load(open('results/peft_sweep_val.json'))
TOP       = 3

top = ranked_cells(sweep['cells'], sweep['adequacy_margin'])[:TOP]
missing = []
for r in top:
    p = sweep_dir / f"{r['tag']}_val.jsonl"
    n = sum(1 for line in p.open() if line.strip()) if p.exists() else 0
    print(f"  {r['tag']:<24} {n:>5}/{n_val}  {p}")
    if n != n_val:
        missing.append(r['tag'])
assert not missing, (
    f'{missing} would be generated under the COMET stack; go back to Phase 1 and finish them'
)
print('\nall top candidates present on full val -- Phase 2 is scoring only')

  peft_r32_lr2e-4_e2        1323/1323  outputs/peft_sweep/peft_r32_lr2e-4_e2_val.jsonl
  peft_r16_lr2e-4_e2        1323/1323  outputs/peft_sweep/peft_r16_lr2e-4_e2_val.jsonl
  peft_r16_lr1e-4_e2        1323/1323  outputs/peft_sweep/peft_r16_lr1e-4_e2_val.jsonl

all top candidates present on full val -- Phase 2 is scoring only


In [15]:
!pip install -q -r requirements-comet.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [14]:
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

import logging
logging.getLogger('httpx').setLevel(logging.WARNING)   # one INFO line per judge call otherwise

In [19]:
!USE_TF=0 python -m src.peft.verify --config configs/peft_sweep.yaml \
    --judge-config configs/judge_eval.yaml --top 3 --comet-adequacy 0.01

Confirming top 3 proxy candidates from peft_sweep_val.json (['peft_r32_lr2e-4_e2', 'peft_r16_lr2e-4_e2', 'peft_r16_lr1e-4_e2']) on val
skip peft_r32_lr2e-4_e2: outputs/peft_sweep/peft_r32_lr2e-4_e2_val.jsonl exists (use --overwrite to regenerate)
skip peft_r16_lr2e-4_e2: outputs/peft_sweep/peft_r16_lr2e-4_e2_val.jsonl exists (use --overwrite to regenerate)
skip peft_r16_lr1e-4_e2: outputs/peft_sweep/peft_r16_lr1e-4_e2_val.jsonl exists (use --overwrite to regenerate)
Fetching 5 files: 100%|██████████████████████| 5/5 [00:00<00:00, 130257.89it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/pytorch_lightning

In [16]:
# The freeze decision on the reported metrics.
import json
v = json.load(open('results/peft_verify_val.json'))
print('proxy pick :', v['proxy_pick'])
print('freeze     :', v['freeze'],
      '(proxy pick held)' if v['proxy_pick_held'] else '(runner-up overtook the proxy pick)')
print()
print(f"{'tag':<24} {'r':>3} {'lr':>7} {'ep':>3} {'chrF':>6} {'COMET':>8} {'Phi':>6} {'cov':>6}")
print('-' * 70)
for c in sorted(v['cells'], key=lambda c: c['comet_system'], reverse=True):
    mark = '  <== freeze' if c['tag'] == v['freeze'] else ''
    phi = f"{c['judge_mean']:.3f}" if c['judge_mean'] is not None else 'n/a'
    cov = f"{c['judge_coverage']:.0%}" if c['judge_coverage'] is not None else 'n/a'
    print(f"{c['tag']:<24} {c['r']:>3} {c['lr']:>7g} {c['epoch']:>3} {c['chrF']:>6} "
          f"{c['comet_system']:>8.4f} {phi:>6} {cov:>6}{mark}")
print('\nfreeze checkpoint:', v['freeze_checkpoint'])

proxy pick : peft_r32_lr2e-4_e2
freeze     : peft_r32_lr2e-4_e2 (proxy pick held)

tag                        r      lr  ep   chrF    COMET    Phi    cov
----------------------------------------------------------------------
peft_r16_lr2e-4_e2        16  0.0002   2  41.73   0.7016  2.732   100%
peft_r32_lr2e-4_e2        32  0.0002   2  41.58   0.6986  2.741   100%  <== freeze
peft_r16_lr1e-4_e2        16  0.0001   2  41.66   0.6978  2.699   100%

freeze checkpoint: models/peft_lora_r32_lr2e-4/checkpoint-1358


### Freeze the adapter into `configs/peft_qwen.yaml`

In [17]:
import json, re, pathlib

v = json.load(open('results/peft_verify_val.json'))
frozen = next(c for c in v['cells'] if c['tag'] == v['freeze'])
ckpt = frozen['checkpoint']
assert pathlib.Path(ckpt, 'adapter_config.json').exists(), f'no adapter at {ckpt}'

p = pathlib.Path('configs/peft_qwen.yaml')
text = p.read_text(encoding='utf-8')
text = re.sub(r'(?m)^(\s*adapter_path:\s*)\S+', lambda m: f'{m.group(1)}{ckpt}', text, count=1)
text = re.sub(r'(?m)^(\s{4}r:\s*)\S+',     lambda m: f"{m.group(1)}{frozen['r']}", text, count=1)
text = re.sub(r'(?m)^(\s{4}alpha:\s*)\S+', lambda m: f"{m.group(1)}{frozen['alpha']}", text, count=1)
text = re.sub(r'(?m)^(\s{4}learning_rate:\s*)\S+',
              lambda m: f"{m.group(1)}{frozen['lr']:g}", text, count=1)
p.write_text(text, encoding='utf-8')

print(f"froze adapter_path={ckpt} (r={frozen['r']}, alpha={frozen['alpha']}, "
      f"lr={frozen['lr']:g}, epoch={frozen['epoch']}) into configs/peft_qwen.yaml")
!grep -nE '^\s*(adapter_path|r|alpha|learning_rate):' configs/peft_qwen.yaml

froze adapter_path=models/peft_lora_r32_lr2e-4/checkpoint-1358 (r=32, alpha=64, lr=0.0002, epoch=2) into configs/peft_qwen.yaml
5:  adapter_path: models/peft_lora_r32_lr2e-4/checkpoint-1358   # LoRA adapter loaded at inference; null = base only
23:    r: 32                               # rank (dev-tuned)
24:    alpha: 64
33:    learning_rate: 0.0002               # dev-tuned


---
## Phase 3 -- generation runtime - the PEFT rung in the ladder

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y torchvision torchaudio

In [ ]:
# Phase 3 context + guard: the prompting rungs must be present and full-val, or the
# ladder comparison is broken.
from pathlib import Path

LADDER = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full']
CONDS  = LADDER + ['peft']
CONDS_ARG = ' '.join(CONDS)
n_val = sum(1 for line in open('data/splits/val.jsonl') if line.strip())

for c in LADDER:
    p = Path(f'outputs/{c}_val.jsonl')
    n = sum(1 for line in p.open() if line.strip()) if p.exists() else 0
    print(f'  {c:<16} {n:>5}/{n_val}')
    assert n == n_val, f'{p} is missing or not full-val; re-run the AFSP full run first'
print('\nprompting ladder intact')

In [ ]:
# Generate the peft rung on full val (resumable: appends into outputs/peft_val.jsonl).
!python3 manage.py infer --condition peft --config configs/peft_qwen.yaml

In [ ]:
import json
from pathlib import Path

freeze_tag = json.load(open('results/peft_verify_val.json'))['freeze']
a = [json.loads(x) for x in Path('outputs/peft_val.jsonl').read_text().splitlines() if x.strip()]
b = [json.loads(x) for x in
     Path(f'outputs/peft_sweep/{freeze_tag}_val.jsonl').read_text().splitlines() if x.strip()]
assert [r['input'] for r in a] == [r['input'] for r in b], 'row order differs between the two files'
diff = [i for i, (x, y) in enumerate(zip(a, b)) if x['prediction'] != y['prediction']]
print(f'{len(a) - len(diff)}/{len(a)} predictions identical to the sweep generation of {freeze_tag}')
if diff:
    print(f'  {len(diff)} segments drifted, first at index {diff[0]}')
    print(f'  infer : {a[diff[0]]["prediction"][:160]}')
    print(f'  sweep : {b[diff[0]]["prediction"][:160]}')

In [ ]:
# Adequacy proxies (chrF/BLEU) and register stylometrics -- free/local, no COMET.
!python manage.py eval         --conditions {CONDS_ARG} --split val
!python manage.py stylometrics --conditions {CONDS_ARG} --split val --targets-split val

In [ ]:
import os, getpass, logging
from pathlib import Path
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)

# What the cache already covers (these resume for free); anything short is a paid pass.
for c in CONDS:
    p = Path(f'results/judge_val_segments/{c}.jsonl')
    n = sum(1 for line in p.open() if line.strip()) if p.exists() else 0
    print(f'  {c:<16} cached {n:>5}/{n_val}')

In [ ]:
!python manage.py judge --conditions {CONDS_ARG} --split val --config configs/judge_eval.yaml

---
## Phase 4 -- COMET runtime - ladder COMET + paired bootstrap

`manage.py comet` rewrites `results/comet_val.json` with only the conditions passed,
and the paired bootstrap needs every condition's per-segment vector in that one file --
so all six are rescored here even though five of them were scored in the AFSP run.
COMET is local and free; this is cheaper than reasoning about a merge.

In [ ]:
!pip install -q -r requirements-comet.txt

In [ ]:
# Phase 4 context (re-derived from disk; safe after a runtime restart).
CONDS = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full', 'peft']
CONDS_ARG = ' '.join(CONDS)
n_val = sum(1 for line in open('data/splits/val.jsonl') if line.strip())
print(CONDS_ARG, '| n_val =', n_val)

In [ ]:
!python manage.py comet --conditions {CONDS_ARG} --split val

In [ ]:
# The bootstrap resamples pairs by index, so verify both metric files carry all six
# conditions, at equal n, over identical sources before trusting any interval.
import json

for metric in ('comet', 'judge'):
    stored = json.load(open(f'results/{metric}_val.json'))
    have = [c for c in CONDS if c in stored]
    missing = [c for c in CONDS if c not in stored]
    assert not missing, f'{metric}: {missing} missing -- rerun that pass over all six conditions'
    ns   = {c: stored[c]['n'] for c in have}
    srcs = {c: stored[c].get('sources') for c in have}
    aligned = all(srcs[c] == srcs[have[0]] for c in have)
    print(f'{metric:<6} conditions {len(have)}/{len(CONDS)}  n={sorted(set(ns.values()))}  '
          f'aligned={aligned}')
    assert set(ns.values()) == {n_val}, f'{metric}: mixed segment counts {ns}'
    assert aligned, f'{metric}: sources differ across conditions; paired bootstrap invalid'
print('\nboth metric files are paired-bootstrap ready')

In [ ]:
# Ladder order matters: --adjacent adds each consecutive pair, so listing peft last buys
# the afsp_full -> peft contrast (prompting's best vs parameter adaptation) on top of
# every rung-vs-zeroshot comparison.
!python manage.py bootstrap --metric comet --conditions {CONDS_ARG} --split val --adjacent

In [ ]:
!python manage.py bootstrap --metric judge --conditions {CONDS_ARG} --split val --adjacent

In [ ]:
# The val table, assembled from the artifacts this run wrote.
import json
from pathlib import Path
from src.eval._io import load_condition
from src.eval.quick import score as quick_score
from src.eval.stylometrics import aggregate, distance_to_centroid

centroid = json.load(open('results/stylometrics_centroid.json'))
comet    = json.load(open('results/comet_val.json'))
judge    = json.load(open('results/judge_val.json'))

print(f"{'condition':<16} {'n':>5} {'chrF':>7} {'BLEU':>7} {'COMET':>8} {'Phi':>6} {'stylo_dist':>11}")
print('-' * 66)
for c in CONDS:
    s = quick_score(c, Path('outputs'), 'val')
    _, preds, _ = load_condition('outputs', c, 'val')
    d = distance_to_centroid(aggregate(preds)['mean'], centroid)
    phi = judge[c]['mean']
    print(f"{c:<16} {s['n']:>5} {s['chrF']:>7.2f} {s['BLEU']:>7.2f} "
          f"{comet[c]['system']:>8.4f} {phi:>6.3f} {d:>11.4f}")

v = json.load(open('results/peft_verify_val.json'))
print(f"\npeft rung = {v['freeze']}  ->  {v['freeze_checkpoint']}")
print('reference marker_rate (gold val targets):',
      quick_score(CONDS[0], Path('outputs'), 'val')['ref_marker_rate'])

In [ ]:
!git status --short outputs results configs